In [ ]:
import os
os.chdir("/content")
print(os.getcwd())

import subprocess
import sys
import os
import time

print("Installing dependencies...")

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "numpy", "opencv-python", "opencv-python-headless", "opencv-contrib-python"],
               stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL)
subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-binary=numpy", "numpy==1.26.4"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "opencv-python-headless==4.9.0.80"])

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "peft", "diffusers", "transformers", "huggingface-hub"],
               stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL)

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "streamlit", "pyngrok"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch==2.5.1", "torchvision==0.20.1", "torchaudio==2.5.1", "--index-url", "https://download.pytorch.org/whl/cu121"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "xformers==0.0.28.post3"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes==0.45.0"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "peft==0.13.2"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "accelerate==1.2.1"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers==4.46.3"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "diffusers==0.27.2"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "safetensors==0.4.5"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface-hub", "--upgrade"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pillow", "--upgrade"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "google-genai"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "facexlib", "basicsr", "realesrgan"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gfpgan"])

print("Dependencies installed!")

import shutil
shutil.rmtree("/content/sd-scripts", ignore_errors=True)

print("Cloning sd-scripts...")

def safe_clone(url):
    return os.system(f"git clone --depth=1 -q {url} /content/sd-scripts")

mirrors = [
    "https://github.com/kohya-ss/sd-scripts.git",
    "https://ghproxy.com/https://github.com/kohya-ss/sd-scripts.git",
    "https://gitclone.com/github.com/kohya-ss/sd-scripts.git",
    "https://hub.nuaa.cf/kohya-ss/sd-scripts.git"
]

clone_success = False
for url in mirrors:
    if safe_clone(url) == 0:
        clone_success = True
        break
    time.sleep(2)
    shutil.rmtree("/content/sd-scripts", ignore_errors=True)

if not clone_success:
    print("All mirrors failed. Downloading ZIP instead...")
    os.system("wget -q https://github.com/kohya-ss/sd-scripts/archive/refs/heads/master.zip -O sd.zip")
    os.system("unzip -q sd.zip -d /content/")
    os.system("mv /content/sd-scripts-master /content/sd-scripts")
    if os.path.exists("/content/sd-scripts"):
        clone_success = True

if not clone_success:
    raise Exception("FAILED: Could not clone sd-scripts.")

os.chdir("/content/sd-scripts")
print("Installing sd-scripts requirements...")
os.system("pip install -q -r requirements.txt")
print("sd-scripts ready!")

os.makedirs(os.path.expanduser("~/.cache/huggingface/accelerate"), exist_ok=True)
with open(os.path.expanduser("~/.cache/huggingface/accelerate/default_config.yaml"), 'w') as f:
    f.write("""compute_environment: LOCAL_MACHINE
distributed_type: 'NO'
downcast_bf16: 'no'
gpu_ids: all
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 1
rdzv_backend: static
same_network: true
tpu_env: []
tpu_use_cluster: false
tpu_use_sudo: false
use_cpu: false
""")

for d in ["/content/models", "/content/training_images", "/content/train_data",
          "/content/output", "/content/logs", "/content/generated"]:
    os.makedirs(d, exist_ok=True)

import re as _re

IDENTITY_ONLY_RE = [
    r"\b(blonde?|brunette|redhead|auburn|chestnut|platinum|silver)[- ]?(hair(ed)?|locks?|curls?|waves?)?\b",
    r"\b(long|short|medium|shoulder[- ]length)[- ]hair(ed)?\b",
    r"\b(straight|curly|wavy|braided|coiled)[- ]hair(ed)?\b",
    r"\b(blue|green|brown|hazel|amber|grey|gray|honey)[- ]?(eyes?|irises?)\b",
    r"\b(fair|pale|olive|tan(ned)?)[- ]?(skin(ned)?|complexion)?\b",
    r"\b(oval|round|square|heart[- ]shaped)[- ]face\b",
    r"\b(full|thin|pouty)[- ]lips?\b",
    r"\b(mediterranean|caucasian|african|latin[ao]?|hispanic|european|"
      r"middle[- ]eastern|south[- ]asian|east[- ]asian)[- ]?(descent|looking)?\b",
]

def strip_conflicting_identity(p: str) -> str:
    for pat in IDENTITY_ONLY_RE:
        p = _re.sub(pat, "", p, flags=_re.IGNORECASE)
    p = _re.sub(r"[ ,]{2,}", ", ", p)
    p = _re.sub(r"^[ ,]+|[ ,]+$", "", p)
    return p.strip()


app_code = r'''
import streamlit as st
import os, shutil, glob, subprocess, time, re
from PIL import Image, ImageFilter, ImageEnhance
import numpy as np
import torch
from diffusers import (StableDiffusionPipeline, StableDiffusionImg2ImgPipeline,
                       AutoencoderKL, DPMSolverMultistepScheduler)
from huggingface_hub import hf_hub_download
from google import genai

st.set_page_config(
    page_title="FrameForge — AI Storyboard Studio",
    layout="wide",
    initial_sidebar_state="collapsed"
)

st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Cormorant+Garamond:ital,wght@0,300;0,400;0,500;0,600;1,300;1,400&family=Syne:wght@400;500;600;700;800&family=IBM+Plex+Mono:wght@300;400;500&display=swap');

:root {
    --bg:         #0a0a0d;
    --bg1:        #0f0f14;
    --bg2:        #141419;
    --bg3:        #1a1a22;
    --border:     #1e1e28;
    --border2:    #2a2a38;
    --accent:     #b8966e;
    --accent2:    #d4b48a;
    --accent3:    #8a6a44;
    --muted:      #4a4a60;
    --muted2:     #6a6a88;
    --text:       #c8c8d8;
    --text2:      #e8e8f4;
    --text3:      #f4f4fc;
    --red:        #a05050;
    --green:      #507860;
    --cyan:       #507888;
    --tag-w:      rgba(80,120,136,0.15);
    --tag-m:      rgba(184,150,110,0.15);
    --tag-c:      rgba(80,120,96,0.15);
}

*, *::before, *::after { box-sizing: border-box; }

html, body, [class*="css"] {
    font-family: 'Syne', sans-serif;
    background-color: var(--bg) !important;
    color: var(--text) !important;
    -webkit-font-smoothing: antialiased;
}

.stApp { background: var(--bg) !important; }
.main .block-container {
    padding: 0 3.5rem 4rem 3.5rem !important;
    max-width: 1480px !important;
}

/* ── Masthead ── */
.masthead {
    padding: 3.5rem 0 2rem 0;
    border-bottom: 1px solid var(--border2);
    margin-bottom: 2.5rem;
    display: flex;
    align-items: flex-end;
    justify-content: space-between;
}
.masthead-left {}
.wordmark {
    font-family: 'Cormorant Garamond', serif;
    font-weight: 300;
    font-size: 4.2rem;
    letter-spacing: 0.06em;
    color: var(--text3);
    line-height: 1;
    margin-bottom: 0.35rem;
}
.wordmark span { color: var(--accent); }
.tagline {
    font-family: 'IBM Plex Mono', monospace;
    font-size: 0.65rem;
    letter-spacing: 0.28em;
    text-transform: uppercase;
    color: var(--muted2);
}
.masthead-right {
    display: flex;
    flex-direction: column;
    align-items: flex-end;
    gap: 6px;
    padding-bottom: 4px;
}
.build-tag {
    font-family: 'IBM Plex Mono', monospace;
    font-size: 0.6rem;
    letter-spacing: 0.18em;
    text-transform: uppercase;
    color: var(--accent3);
    border: 1px solid var(--accent3);
    padding: 4px 12px;
    border-radius: 1px;
}
.gpu-indicator {
    display: flex;
    align-items: center;
    gap: 6px;
    font-family: 'IBM Plex Mono', monospace;
    font-size: 0.6rem;
    letter-spacing: 0.14em;
    color: var(--muted2);
}
.gpu-dot {
    width: 6px; height: 6px;
    border-radius: 50%;
    background: var(--green);
    box-shadow: 0 0 8px rgba(80,120,96,0.8);
}

/* ── Info strip ── */
.info-strip {
    background: var(--bg1);
    border: 1px solid var(--border2);
    border-left: 2px solid var(--accent3);
    padding: 14px 20px;
    margin-bottom: 2.5rem;
    display: flex;
    align-items: center;
    gap: 20px;
}
.info-strip-label {
    font-family: 'IBM Plex Mono', monospace;
    font-size: 0.6rem;
    letter-spacing: 0.22em;
    text-transform: uppercase;
    color: var(--accent);
    white-space: nowrap;
}
.info-strip-content {
    font-family: 'IBM Plex Mono', monospace;
    font-size: 0.68rem;
    color: var(--muted2);
    letter-spacing: 0.06em;
    line-height: 1.7;
}

/* ── Tabs ── */
.stTabs [data-baseweb="tab-list"] {
    gap: 0 !important;
    background: transparent !important;
    border: none !important;
    border-bottom: 1px solid var(--border2) !important;
    padding: 0 !important;
    margin-bottom: 2.5rem !important;
}
.stTabs [data-baseweb="tab"] {
    font-family: 'IBM Plex Mono', monospace !important;
    font-size: 0.65rem !important;
    letter-spacing: 0.22em !important;
    text-transform: uppercase !important;
    color: var(--muted2) !important;
    background: transparent !important;
    border: none !important;
    border-bottom: 2px solid transparent !important;
    padding: 14px 28px !important;
    margin-bottom: -1px !important;
    transition: all 0.2s !important;
}
.stTabs [data-baseweb="tab"]:hover { color: var(--text) !important; }
.stTabs [aria-selected="true"] {
    color: var(--accent) !important;
    border-bottom-color: var(--accent) !important;
}
.stTabs [data-baseweb="tab-highlight"],
.stTabs [data-baseweb="tab-border"] { display: none !important; }

/* ── Section headings ── */
.section-heading {
    font-family: 'Cormorant Garamond', serif;
    font-weight: 400;
    font-size: 1.9rem;
    letter-spacing: 0.04em;
    color: var(--text3);
    margin-bottom: 0.25rem;
    line-height: 1.1;
}
.section-sub {
    font-family: 'IBM Plex Mono', monospace;
    font-size: 0.62rem;
    letter-spacing: 0.22em;
    text-transform: uppercase;
    color: var(--muted);
    margin-bottom: 2rem;
}

/* ── Panel ── */
.panel {
    background: var(--bg1);
    border: 1px solid var(--border2);
    padding: 1.8rem;
    margin-bottom: 1.2rem;
}
.panel-title {
    font-family: 'IBM Plex Mono', monospace;
    font-size: 0.6rem;
    letter-spacing: 0.26em;
    text-transform: uppercase;
    color: var(--accent);
    padding-bottom: 12px;
    border-bottom: 1px solid var(--border);
    margin-bottom: 1.4rem;
}

/* ── Form elements ── */
.stTextInput > div > div > input,
.stTextArea > div > div > textarea,
.stNumberInput > div > div > input {
    background: var(--bg2) !important;
    border: 1px solid var(--border2) !important;
    border-radius: 0 !important;
    color: var(--text2) !important;
    font-family: 'Syne', sans-serif !important;
    font-size: 0.88rem !important;
    font-weight: 400 !important;
    padding: 11px 14px !important;
    transition: border-color 0.15s !important;
    box-shadow: none !important;
}
.stTextInput > div > div > input:focus,
.stTextArea > div > div > textarea:focus {
    border-color: var(--accent3) !important;
    box-shadow: none !important;
    outline: none !important;
}

label,
.stTextInput label,
.stTextArea label,
.stSelectbox label,
.stSlider label,
.stCheckbox label,
.stFileUploader label {
    font-family: 'IBM Plex Mono', monospace !important;
    font-size: 0.6rem !important;
    letter-spacing: 0.2em !important;
    text-transform: uppercase !important;
    color: var(--muted2) !important;
    margin-bottom: 6px !important;
    font-weight: 400 !important;
}

/* ── Selectbox ── */
.stSelectbox > div > div,
.stSelectbox [data-baseweb="select"] > div {
    background: var(--bg2) !important;
    border-color: var(--border2) !important;
    border-radius: 0 !important;
    color: var(--text2) !important;
    font-family: 'Syne', sans-serif !important;
    font-size: 0.88rem !important;
    box-shadow: none !important;
}
.stSelectbox [data-baseweb="select"] > div:focus-within {
    border-color: var(--accent3) !important;
    box-shadow: none !important;
}

/* ── Slider ── */
.stSlider > div > div > div > div { background: var(--accent) !important; }
.stSlider [data-testid="stThumbValue"] {
    font-family: 'IBM Plex Mono', monospace !important;
    font-size: 0.65rem !important;
    color: var(--accent) !important;
    background: var(--bg2) !important;
    border: 1px solid var(--border2) !important;
    padding: 2px 7px !important;
    border-radius: 1px !important;
}
[data-testid="stSlider"] > div > div {
    background: var(--border2) !important;
    border-radius: 1px !important;
}

/* ── Buttons ── */
.stButton > button {
    font-family: 'IBM Plex Mono', monospace !important;
    font-size: 0.65rem !important;
    letter-spacing: 0.22em !important;
    text-transform: uppercase !important;
    background: transparent !important;
    border: 1px solid var(--border2) !important;
    color: var(--muted2) !important;
    padding: 13px 32px !important;
    border-radius: 0 !important;
    transition: all 0.18s !important;
    font-weight: 400 !important;
}
.stButton > button:hover {
    border-color: var(--accent) !important;
    color: var(--accent) !important;
    background: rgba(184,150,110,0.04) !important;
    box-shadow: none !important;
}
.stButton > button[kind="primary"] {
    background: var(--accent) !important;
    border-color: var(--accent) !important;
    color: var(--bg) !important;
    font-weight: 500 !important;
}
.stButton > button[kind="primary"]:hover {
    background: var(--accent2) !important;
    border-color: var(--accent2) !important;
    box-shadow: none !important;
}
.stDownloadButton > button {
    font-family: 'IBM Plex Mono', monospace !important;
    font-size: 0.6rem !important;
    letter-spacing: 0.18em !important;
    text-transform: uppercase !important;
    background: var(--bg2) !important;
    border: 1px solid var(--border2) !important;
    color: var(--muted2) !important;
    padding: 9px 20px !important;
    border-radius: 0 !important;
    transition: all 0.18s !important;
    width: 100% !important;
}
.stDownloadButton > button:hover {
    border-color: var(--accent3) !important;
    color: var(--accent) !important;
    background: rgba(184,150,110,0.06) !important;
}

/* ── File uploader ── */
[data-testid="stFileUploader"] > div {
    background: var(--bg1) !important;
    border: 1px dashed var(--border2) !important;
    border-radius: 0 !important;
    padding: 2rem !important;
    transition: border-color 0.18s !important;
}
[data-testid="stFileUploader"] > div:hover { border-color: var(--accent3) !important; }
[data-testid="stFileUploader"] p {
    color: var(--muted) !important;
    font-size: 0.8rem !important;
    font-family: 'IBM Plex Mono', monospace !important;
}

/* ── Checkbox ── */
.stCheckbox > label {
    text-transform: none !important;
    font-family: 'Syne', sans-serif !important;
    font-size: 0.85rem !important;
    letter-spacing: 0 !important;
    color: var(--text) !important;
    font-weight: 400 !important;
}
[data-testid="stCheckbox"] input:checked + span {
    background: var(--accent) !important;
    border-color: var(--accent) !important;
}

/* ── Alerts ── */
.stAlert > div {
    border-radius: 0 !important;
    font-family: 'IBM Plex Mono', monospace !important;
    font-size: 0.72rem !important;
    letter-spacing: 0.04em !important;
    border-left-width: 2px !important;
}
[data-testid="stInfo"] > div {
    background: rgba(80,120,136,0.08) !important;
    border-left-color: var(--cyan) !important;
    color: var(--cyan) !important;
}
[data-testid="stWarning"] > div {
    background: rgba(184,150,110,0.08) !important;
    border-left-color: var(--accent3) !important;
    color: var(--accent2) !important;
}
[data-testid="stSuccess"] > div {
    background: rgba(80,120,96,0.08) !important;
    border-left-color: var(--green) !important;
    color: #6aaa88 !important;
}
[data-testid="stError"] > div,
.stException > div {
    background: rgba(160,80,80,0.08) !important;
    border-left-color: var(--red) !important;
    color: #c07070 !important;
}

/* ── Progress ── */
.stProgress > div > div > div > div {
    background: linear-gradient(90deg, var(--accent3), var(--accent)) !important;
    border-radius: 0 !important;
}
.stProgress > div > div > div {
    background: var(--bg3) !important;
    border-radius: 0 !important;
    height: 2px !important;
}

/* ── Expander ── */
.streamlit-expanderHeader {
    font-family: 'IBM Plex Mono', monospace !important;
    font-size: 0.6rem !important;
    letter-spacing: 0.18em !important;
    text-transform: uppercase !important;
    color: var(--muted2) !important;
    background: var(--bg1) !important;
    border: 1px solid var(--border2) !important;
    border-radius: 0 !important;
    padding: 11px 16px !important;
    transition: all 0.15s !important;
}
.streamlit-expanderHeader:hover {
    border-color: var(--accent3) !important;
    color: var(--accent) !important;
}
.streamlit-expanderContent {
    background: var(--bg1) !important;
    border: 1px solid var(--border2) !important;
    border-top: none !important;
    border-radius: 0 !important;
    padding: 1.2rem !important;
}

/* ── Images ── */
[data-testid="stImage"] img {
    border-radius: 0 !important;
    border: 1px solid var(--border2) !important;
}
[data-testid="column"] { padding: 0 0.6rem !important; }
[data-testid="column"]:first-child { padding-left: 0 !important; }
[data-testid="column"]:last-child { padding-right: 0 !important; }

/* ── Code ── */
.stCodeBlock, code {
    font-family: 'IBM Plex Mono', monospace !important;
    font-size: 0.72rem !important;
    background: var(--bg2) !important;
    border: 1px solid var(--border2) !important;
    border-radius: 0 !important;
    color: #8aaf8a !important;
}

/* ── Metric ── */
[data-testid="stMetric"] {
    background: var(--bg2) !important;
    border: 1px solid var(--border2) !important;
    padding: 12px 16px !important;
}
[data-testid="stMetricLabel"] {
    font-family: 'IBM Plex Mono', monospace !important;
    font-size: 0.58rem !important;
    letter-spacing: 0.2em !important;
    text-transform: uppercase !important;
    color: var(--muted2) !important;
}
[data-testid="stMetricValue"] {
    font-family: 'Syne', sans-serif !important;
    font-size: 1.4rem !important;
    font-weight: 600 !important;
    color: var(--accent) !important;
}

hr {
    border: none !important;
    border-top: 1px solid var(--border2) !important;
    margin: 2.5rem 0 !important;
}

/* ── Scene card ── */
.scene-card {
    background: var(--bg1);
    border: 1px solid var(--border2);
    border-top: 2px solid var(--accent3);
    padding: 1.6rem 1.8rem 1.4rem;
    margin-bottom: 0.5rem;
    position: relative;
}
.scene-watermark {
    font-family: 'Cormorant Garamond', serif;
    font-size: 7rem;
    font-weight: 300;
    color: rgba(184,150,110,0.04);
    position: absolute;
    top: -8px;
    right: 16px;
    line-height: 1;
    pointer-events: none;
    letter-spacing: -0.02em;
    user-select: none;
}
.scene-eyebrow {
    font-family: 'IBM Plex Mono', monospace;
    font-size: 0.58rem;
    letter-spacing: 0.26em;
    text-transform: uppercase;
    color: var(--accent3);
    margin-bottom: 10px;
    display: flex;
    align-items: center;
    gap: 12px;
}
.scene-description {
    font-family: 'Cormorant Garamond', serif;
    font-size: 1.05rem;
    font-style: italic;
    color: var(--muted2);
    line-height: 1.7;
    margin-bottom: 10px;
    font-weight: 300;
}
.scene-mood-line {
    font-family: 'IBM Plex Mono', monospace;
    font-size: 0.6rem;
    letter-spacing: 0.16em;
    text-transform: uppercase;
    color: var(--muted);
}

/* ── Shot tags ── */
.shot-tag {
    display: inline-block;
    font-family: 'IBM Plex Mono', monospace;
    font-size: 0.55rem;
    letter-spacing: 0.18em;
    text-transform: uppercase;
    padding: 3px 10px;
    border: 1px solid;
}
.shot-wide  { color: #507888; border-color: rgba(80,120,136,0.35); background: var(--tag-w); }
.shot-med   { color: #b8966e; border-color: rgba(184,150,110,0.35); background: var(--tag-m); }
.shot-close { color: #507860; border-color: rgba(80,120,96,0.35);  background: var(--tag-c); }
.shot-mood  { color: var(--muted); border-color: var(--border2); background: var(--bg3); }

/* ── Storyboard grid: 2 per row ── */
.sb-row {
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 1.2rem;
    margin-bottom: 1.2rem;
}
.sb-cell {
    background: var(--bg1);
    border: 1px solid var(--border2);
    overflow: hidden;
}
.sb-cell img {
    width: 100%;
    display: block;
    border: none !important;
}
.sb-caption {
    padding: 10px 14px 12px;
    display: flex;
    justify-content: space-between;
    align-items: center;
}
.sb-caption-label {
    font-family: 'IBM Plex Mono', monospace;
    font-size: 0.58rem;
    letter-spacing: 0.18em;
    text-transform: uppercase;
    color: var(--muted2);
}
.sb-caption-meta {
    font-family: 'IBM Plex Mono', monospace;
    font-size: 0.55rem;
    letter-spacing: 0.12em;
    color: var(--muted);
}

/* ── Dividers ── */
.section-rule {
    border: none;
    border-top: 1px solid var(--border);
    margin: 2rem 0;
}

/* ── Scrollbar ── */
::-webkit-scrollbar { width: 4px; height: 4px; }
::-webkit-scrollbar-track { background: var(--bg); }
::-webkit-scrollbar-thumb { background: var(--border2); border-radius: 0; }
::-webkit-scrollbar-thumb:hover { background: var(--accent3); }
</style>
""", unsafe_allow_html=True)

# ── Gemini client
client = genai.Client(api_key="XXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")

# ── Identity stripping
IDENTITY_ONLY_RE = [
    r"\b(blonde?|brunette|redhead|auburn|chestnut|platinum|silver)[- ]?(hair(ed)?|locks?|curls?|waves?)?\b",
    r"\b(long|short|medium|shoulder[- ]length)[- ]hair(ed)?\b",
    r"\b(straight|curly|wavy|braided|coiled)[- ]hair(ed)?\b",
    r"\b(blue|green|brown|hazel|amber|grey|gray|honey)[- ]?(eyes?|irises?)\b",
    r"\b(fair|pale|olive|tan(ned)?)[- ]?(skin(ned)?|complexion)?\b",
    r"\b(oval|round|square|heart[- ]shaped)[- ]face\b",
    r"\b(full|thin|pouty)[- ]lips?\b",
    r"\b(mediterranean|caucasian|african|latin[ao]?|hispanic|european|"
      r"middle[- ]eastern|south[- ]asian|east[- ]asian)[- ]?(descent|looking)?\b",
]

def strip_conflicting_identity(p: str) -> str:
    for pat in IDENTITY_ONLY_RE:
        p = re.sub(pat, "", p, flags=re.IGNORECASE)
    p = re.sub(r"[ ,]{2,}", ", ", p).strip().strip(",").strip()
    return p

# ── Shot detection
WIDE_KW = ["wide shot","wide angle","establishing","full body","full-body","long shot",
           "aerial","panoramic","landscape","environment","running","walking","fighting",
           "riding","jumping","fleeing","chasing","battlefield","crowd","arena","vast",
           "horizon","cityscape","kingdom","background","scenery","epic scene","epic"]
MED_KW  = ["medium shot","waist up","waist-up","three quarter","3/4","standing",
           "reaching","holding","drawing sword","gesture","action"]

def detect_frame(prompt):
    low = prompt.lower()
    for k in WIDE_KW:
        if k in low: return "wide"
    for k in MED_KW:
        if k in low: return "medium"
    return "close"

def framing_tags(frame):
    if frame == "wide":
        return (
            "cinematic wide establishing shot, rich detailed environment, "
            "dramatic atmospheric lighting, sharp foreground character, "
            "detailed background, epic scope, depth of field",
            "portrait only, cropped background, flat scene, boring composition"
        )
    if frame == "medium":
        return (
            "medium shot waist-up, cinematic composition, "
            "expressive face, dynamic pose, dramatic rim lighting, "
            "sharp focus on character, detailed costume",
            "extreme close-up, blurry background bleed, full-body lost in frame"
        )
    return (
        "close-up portrait, chest-up framing, "
        "tack-sharp facial features, beautiful detailed eyes, "
        "cinematic portrait lighting, subtle rim light, bokeh background, "
        "pore-level skin detail, catchlight in eyes",
        "wide shot, full body, tiny face, distant figure"
    )

def quality_suffix(frame):
    base = ("masterpiece, best quality, photorealistic, film photography, "
            "35mm film grain, natural skin texture, RAW photo, "
            "professional cinematography, Kodak Portra 400")
    if frame == "wide":
        return (base + ", sweeping epic vista, dramatic sky, "
                "volumetric light, environmental storytelling, sharp throughout")
    if frame == "medium":
        return (base + ", dramatic three-point lighting, "
                "cinematic color grade, detailed fabric texture")
    return (base + ", Hasselblad H6D, f/2.8 portrait lens, "
            "soft studio lighting, natural catchlight, sharp focus")

COMMON_NEG = (
    "worst quality, low quality, lowres, blurry, soft focus, out of focus, "
    "bad anatomy, bad hands, deformed, disfigured, extra limbs, missing limbs, "
    "mutated hands, malformed, ugly, duplicate, morbid, watermark, signature, "
    "text, jpeg artifacts, oversaturated, overexposed, flat lighting, "
    "plastic skin, wax skin, wax figure, mannequin, CGI face, airbrushed, "
    "poreless skin, flawless skin, smooth skin, hyperrealistic skin, "
    "painting, cartoon, anime, 3d render, render"
)

# ── GFPGAN
_gfpgan_restorer = None

def get_gfpgan():
    global _gfpgan_restorer
    if _gfpgan_restorer is not None:
        return _gfpgan_restorer
    try:
        from gfpgan import GFPGANer
        _gfpgan_restorer = GFPGANer(
            model_path='https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth',
            upscale=1, arch='clean', channel_multiplier=2, bg_upsampler=None
        )
        return _gfpgan_restorer
    except:
        return None

def restore_faces_gfpgan(pil_image):
    try:
        restorer = get_gfpgan()
        if restorer is None: return pil_image
        import cv2
        img_np  = np.array(pil_image.convert("RGB"))
        img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
        _, _, restored_bgr = restorer.enhance(
            img_bgr, has_aligned=False, only_center_face=False,
            paste_back=True, weight=0.78)
        return Image.fromarray(cv2.cvtColor(restored_bgr, cv2.COLOR_BGR2RGB))
    except:
        return pil_image

def sharpen_face_i2i(image, pipe, char_prompt, neg, steps, device, lora_ok, lscale):
    try:
        import cv2
        w, h  = image.size
        cw    = max(w // 2, 320)
        ch    = int(h * 0.45)
        left  = (w - cw) // 2
        face_crop = image.crop((left, 0, left + cw, ch))
        face_512  = face_crop.resize((512, 512), Image.Resampling.LANCZOS)
        face_pos  = (f"{char_prompt}, natural skin texture, pores visible, "
                     "sharp eyes, cinematic lighting, photorealistic")
        face_neg  = (neg + ", plastic skin, wax skin, mannequin, airbrushed, "
                     "smooth skin, flawless skin, poreless")
        i2i = StableDiffusionImg2ImgPipeline(**pipe.components).to(device)
        kw  = dict(prompt=face_pos, negative_prompt=face_neg, image=face_512,
                   strength=0.18, num_inference_steps=max(25, steps),
                   guidance_scale=6.5, generator=torch.Generator(device).manual_seed(99))
        if lora_ok and lscale > 0:
            kw["cross_attention_kwargs"] = {"scale": lscale}
        with torch.autocast(device):
            sharpened = i2i(**kw).images[0]
        sharpened = sharpened.resize((cw, ch), Image.Resampling.LANCZOS)
        mask    = Image.new("L", (cw, ch), 255)
        feather = max(24, cw // 8)
        mask_np = np.array(mask, dtype=np.float32)
        for i in range(feather):
            alpha = int(255 * (i / feather))
            if i < ch:     mask_np[i, :]     = np.minimum(mask_np[i, :],     alpha)
            if ch-1-i >= 0: mask_np[ch-1-i, :] = np.minimum(mask_np[ch-1-i, :], alpha)
            if i < cw:     mask_np[:, i]     = np.minimum(mask_np[:, i],     alpha)
            if cw-1-i >= 0: mask_np[:, cw-1-i] = np.minimum(mask_np[:, cw-1-i], alpha)
        mask = Image.fromarray(mask_np.astype(np.uint8), mode="L")
        out  = image.copy()
        out.paste(sharpened, (left, 0), mask)
        return out
    except:
        return image

# ── Session state
for k, v in [("trained_lora", None), ("pipe", None),
             ("trigger_word", "ohwx"), ("lora_loaded_path", None),
             ("rendered_scenes", [])]:
    if k not in st.session_state:
        st.session_state[k] = v

# ── Masthead
st.markdown("""
<div class="masthead">
  <div class="masthead-left">
    <div class="wordmark">Frame<span>Forge</span></div>
    <div class="tagline">AI Storyboard Studio &nbsp;&mdash;&nbsp; LoRA Character Training</div>
  </div>
  <div class="masthead-right">
    <div class="build-tag">SD 1.5 &nbsp;&middot;&nbsp; GFPGAN &nbsp;&middot;&nbsp; Gemini</div>
    <div class="gpu-indicator"><div class="gpu-dot"></div>GPU Accelerated</div>
  </div>
</div>
""", unsafe_allow_html=True)

st.markdown("""
<div class="info-strip">
  <div class="info-strip-label">Skin Fidelity</div>
  <div class="info-strip-content">
    GFPGAN weight 0.78 — sharp eyes, natural pores, no wax effect &nbsp;&nbsp;|&nbsp;&nbsp;
    img2img strength 0.18 — grain-level sharpness only, identity preserved &nbsp;&nbsp;|&nbsp;&nbsp;
    Negative prompt blocks: plastic skin, mannequin, airbrushed &nbsp;&nbsp;|&nbsp;&nbsp;
    guidance_scale 6.5 — reduces token over-commitment for photographic output
  </div>
</div>
""", unsafe_allow_html=True)

tab1, tab2 = st.tabs(["Train Character LoRA", "Generate Storyboard"])

# ════════════════ TAB 1 ══════════════════════════════════════════════════════
with tab1:
    st.markdown('<div class="section-heading">Train Character LoRA</div>', unsafe_allow_html=True)
    st.markdown('<div class="section-sub">Upload reference images to encode a character into a LoRA adapter</div>', unsafe_allow_html=True)

    st.warning("SD 1.5 is optimised for 512x512. Use 512 for sharpest face fidelity.")

    col_main, col_cfg = st.columns([3, 2], gap="large")

    with col_main:
        uploaded = st.file_uploader(
            "Character Images (10–20 recommended)",
            type=["png", "jpg", "jpeg", "webp"],
            accept_multiple_files=True,
            key="train_images"
        )
        if uploaded:
            st.success(f"{len(uploaded)} images staged for training")
            thumb_cols = st.columns(min(4, len(uploaded)))
            for col, f in zip(thumb_cols, uploaded[:4]):
                with col:
                    st.image(Image.open(f), caption=f.name, use_container_width=True)

    with col_cfg:
        st.markdown('<div class="panel"><div class="panel-title">Training Configuration</div>', unsafe_allow_html=True)

        tw = st.text_input("Trigger Token", value="ohwx",
            help="Use a rare non-word like 'ohwx'. At generation time write 'ohwx woman'.")
        if any(w in tw.lower() for w in ["person", "woman", "man", "girl", "boy", "human"]):
            st.error("Remove common words from trigger token — use a bare token like 'ohwx'.")

        st.info("At generation time, write 'ohwx woman' or 'ohwx man' in your prompt to steer gender.")

        tr = st.selectbox("Training Resolution", ["512 (recommended)", "768", "1024"], index=0)
        res = int(tr.split()[0])

        steps_t = st.slider("Training Steps", 200, 2000, 800, 100)
        lr      = st.selectbox("Learning Rate", ["1e-4", "5e-5", "1e-5"], index=0)

        c_a, c_b = st.columns(2)
        with c_a:
            ndim   = st.selectbox("Network Dim", [32, 64, 128], index=1)
        with c_b:
            nalpha = ndim // 2
            st.metric("Network Alpha", nalpha)

        st.markdown('</div>', unsafe_allow_html=True)

    st.markdown("<br>", unsafe_allow_html=True)

    if st.button("Begin Training", type="primary", key="train_btn"):
        if not uploaded:
            st.error("Upload training images before starting.")
        else:
            st.session_state.trigger_word    = tw
            st.session_state.pipe            = None
            st.session_state.lora_loaded_path = None
            with st.spinner(f"Training LoRA at {res}x{res}..."):
                for d in ["/content/training_images", "/content/train_data", "/content/output"]:
                    if os.path.exists(d): shutil.rmtree(d)
                    os.makedirs(d)
                pb     = st.progress(0)
                st_txt = st.empty()
                st_txt.text("Resizing images...")
                for i, f in enumerate(uploaded):
                    img = Image.open(f)
                    if img.mode == "RGBA": img = img.convert("RGB")
                    img = img.resize((res, res), Image.Resampling.LANCZOS)
                    img.save(f"/content/training_images/{os.path.splitext(f.name)[0]}.png", "PNG")
                    pb.progress((i + 1) / (len(uploaded) * 5))
                st_txt.text("Downloading base model...")
                try:
                    model_path = hf_hub_download(
                        repo_id="SG161222/Realistic_Vision_V5.1_noVAE",
                        filename="Realistic_Vision_V5.1_fp16-no-ema.safetensors",
                        local_dir="/content/models",
                        token="hf_xxxxxxxxxxxxxxxxxxxxxxxx")
                except Exception:
                    model_path = "/content/models/Realistic_Vision_V5.1_fp16-no-ema.safetensors"
                pb.progress(0.3)
                st_txt.text("Preparing dataset...")
                rp   = 15
                ddir = f"/content/train_data/{rp}_{tw}"
                os.makedirs(ddir, exist_ok=True)
                caps = [
                    f"photo of {tw}", f"portrait of {tw}", f"close-up photo of {tw}",
                    f"headshot of {tw}", f"face of {tw}", f"{tw}, natural lighting",
                    f"{tw}, studio lighting", f"{tw}, sharp focus"
                ]
                for src in glob.glob("/content/training_images/*.*"):
                    shutil.copy(src, ddir)
                for i, img_f in enumerate(sorted(glob.glob(f"{ddir}/*.*"))):
                    if img_f.lower().endswith(('.png','.jpg','.jpeg','.webp')):
                        with open(os.path.splitext(img_f)[0] + ".txt", 'w') as fh:
                            fh.write(caps[i % len(caps)])
                pb.progress(0.4)
                st_txt.text("Training LoRA...")
                os.chdir("/content/sd-scripts")
                tcmd = f"""accelerate launch --num_cpu_threads_per_process=2 train_network.py \
--pretrained_model_name_or_path="{model_path}" \
--train_data_dir="/content/train_data" --output_dir="/content/output" \
--output_name="lora_face" --logging_dir="/content/logs" \
--resolution={res},{res} --train_batch_size=1 --gradient_accumulation_steps=4 \
--max_train_steps={steps_t} --learning_rate={lr} --unet_lr={lr} --text_encoder_lr=5e-5 \
--lr_scheduler=cosine_with_restarts --lr_scheduler_num_cycles=3 --lr_warmup_steps=100 \
--network_module=networks.lora --network_dim={ndim} --network_alpha={nalpha} \
--save_model_as=safetensors --save_precision=fp16 --mixed_precision=fp16 \
--vae=stabilityai/sd-vae-ft-mse \
--optimizer_type=AdamW8bit --max_data_loader_n_workers=0 --clip_skip=2 \
--xformers --gradient_checkpointing --save_every_n_epochs=1 --save_last_n_epochs=3 \
--min_snr_gamma=5 --noise_offset=0.01 --caption_extension=".txt" \
--shuffle_caption --keep_tokens=1 --seed=42"""
                lc = st.expander("Training Logs", expanded=True)
                with lc:
                    lp = st.empty()
                    lf = "/content/training_output.log"
                    with open(lf, 'w') as fh:
                        proc = subprocess.Popen(tcmd, shell=True, stdout=fh,
                                                stderr=subprocess.STDOUT, text=True)
                        while proc.poll() is None:
                            if os.path.exists(lf):
                                with open(lf) as lfh:
                                    lp.code(lfh.read()[-3000:], language="text")
                            time.sleep(2)
                        with open(lf) as lfh:
                            lp.code(lfh.read()[-3000:], language="text")
                    if proc.returncode != 0:
                        st.error(f"Training failed (exit code {proc.returncode})")
                pb.progress(0.95)
                time.sleep(5)
                lfiles = sorted(glob.glob("/content/output/*.safetensors"),
                                key=os.path.getmtime, reverse=True)
                if lfiles:
                    st.session_state.trained_lora = lfiles[0]
                    pb.progress(1.0)
                    st_txt.text("Training complete.")
                    st.success(f"LoRA saved: {os.path.basename(lfiles[0])}")
                    with open(lfiles[0], "rb") as fh:
                        st.download_button(
                            "Download Trained LoRA",
                            data=fh,
                            file_name=os.path.basename(lfiles[0]),
                            mime="application/octet-stream",
                            key="dl_lora"
                        )
                else:
                    st.error("No LoRA file found. Check training logs above.")

# ════════════════ TAB 2 ══════════════════════════════════════════════════════
with tab2:
    st.markdown('<div class="section-heading">Generate Storyboard</div>', unsafe_allow_html=True)
    st.markdown('<div class="section-sub">Script scenes with Gemini, then render each frame with Stable Diffusion</div>', unsafe_allow_html=True)

    col_story, col_cfg = st.columns([3, 2], gap="large")

    with col_story:
        story_title = st.text_input("Story Title", value="The Hero's Journey", key="title")
        story_genre = st.selectbox("Genre",
            ["Fantasy", "Sci-Fi", "Adventure", "Mystery", "Romance", "Horror", "Drama", "Comedy"],
            key="genre")
        story_desc = st.text_area(
            "Story Description",
            value="A young hero discovers hidden powers and must save their world from an ancient evil.",
            height=130, key="desc")
        char_gender = st.selectbox(
            "Character Gender / Role",
            ["woman", "man", "girl", "boy", "warrior", "mage", "knight", "ranger", "custom"],
            index=0,
            help="Appended after trigger token in every prompt."
        )
        if char_gender == "custom":
            char_gender = st.text_input("Custom gender / role", value="warrior", key="custom_gender")

    with col_cfg:
        st.markdown('<div class="panel"><div class="panel-title">Generation Settings</div>', unsafe_allow_html=True)

        use_tr  = st.checkbox("Use trained character LoRA",
                              value=bool(st.session_state.trained_lora), key="use_lora")
        lora_path = None
        char      = f"a {char_gender}"
        lscale    = 0.0

        if use_tr and st.session_state.trained_lora:
            lora_path = st.session_state.trained_lora
            st.info(f"Active LoRA: {os.path.basename(lora_path)}")
            char   = f"{st.session_state.trigger_word} {char_gender}"
            lscale = st.slider("LoRA Strength", 0.0, 1.5, 0.85, 0.05,
                               help="0.8–0.95 recommended. Wide shots auto-boost +0.1.")
        else:
            ul = st.file_uploader("Upload LoRA (.safetensors)",
                                  type=["safetensors"], key="upload_lora")
            if ul:
                lora_path = f"/content/{ul.name}"
                with open(lora_path, "wb") as fh:
                    fh.write(ul.getbuffer())
                st.success(f"Loaded: {ul.name}")
                raw_trigger = st.text_input("Trigger Token (e.g. ohwx)", value="ohwx", key="char_name")
                char   = f"{raw_trigger} {char_gender}"
                lscale = st.slider("LoRA Strength", 0.0, 1.5, 0.85, 0.05, key="ls2")
            else:
                char = f"a {char_gender}"

        num_scenes = st.slider("Number of Scenes", 3, 8, 6, key="scenes")

        quality_sel = st.selectbox("Image Quality",
            ["Fast — 25 steps", "Balanced — 35 steps", "High — 50 steps"], index=1, key="quality")
        steps_g = {"Fast — 25 steps": 25, "Balanced — 35 steps": 35, "High — 50 steps": 50}[quality_sel]

        res_sel = st.selectbox("Resolution",
            ["512x512 — SD 1.5 native, sharpest", "768x512 — cinematic widescreen", "512x768 — portrait"],
            index=0)
        res_map = {
            "512x512 — SD 1.5 native, sharpest": (512, 512),
            "768x512 — cinematic widescreen": (768, 512),
            "512x768 — portrait": (512, 768)
        }
        gen_w, gen_h = res_map[res_sel]

        face_mode = st.selectbox(
            "Face Enhancement",
            ["GFPGAN only — natural skin (recommended)",
             "img2img refine — very subtle",
             "GFPGAN + img2img — maximum sharpness",
             "None — raw SD output"],
            index=0
        )
        do_gfpgan = "GFPGAN" in face_mode
        do_i2i    = "img2img" in face_mode

        scene_focus = st.checkbox("Scene-first prompting — richer environments", value=True)

        st.markdown('</div>', unsafe_allow_html=True)

    st.markdown("<br>", unsafe_allow_html=True)

    if st.button("Generate Storyboard", type="primary", key="gen_btn"):
        if not story_title or not story_desc:
            st.error("Provide a story title and description.")
            st.stop()

        st.session_state.rendered_scenes = []

        with st.spinner("Writing cinematic scenes with Gemini..."):
            try:
                base_prompt = f"""You are a professional film storyboard artist and cinematographer.

Your task is to create a high-quality cinematic storyboard for a {story_genre} story titled:
"{story_title}"

Story Context:
{story_desc}

Main Character Token:
"{char}"

You must generate EXACTLY {num_scenes} scenes.

---

🎬 STRICT CINEMATIC RULES:

1. IMAGE PROMPT STRUCTURE (MANDATORY ORDER):
   {char}(if any), [specific location], [time of day], [weather/atmosphere], [lighting], [character action + emotion], [shot type], cinematic, ultra detailed

2. ALWAYS start Image Prompt with only if needed in the scenes:
   {char},

3. ENVIRONMENT RULES:
   - Use SPECIFIC, named, real-feeling locations (e.g., "abandoned railway station platform", NOT "outside")
   - Avoid generic words like: "place", "area", "location"

4. LIGHTING RULES:
   - Must include cinematic lighting terms:
     (e.g., soft diffused light, harsh shadows, rim lighting, volumetric light, neon glow)

5. CHARACTER RULES:
   - DO NOT describe: hair, skin, eyes, ethnicity
   - Focus only on action, posture, emotion

6. SHOT TYPE DISTRIBUTION (VERY IMPORTANT):
   - Close-up → emotional intensity (face / chest-up)
   - Medium → action or interaction (waist-up)
   - Wide → environment storytelling

7. WIDE SHOT REQUIREMENT:
   Must include EXACT phrase:
   "{char} clearly visible in foreground, recognisable face"

8. MOOD:
   - Each scene MUST have a UNIQUE one-word mood
   - Must match the visual tone (e.g., ominous, serene, tense, euphoric)

9. VARIETY:
   - No repeated environments
   - No repeated lighting styles
   - No repeated shot compositions

10. OUTPUT STYLE:
   - Highly visual, cinematic, not generic storytelling
   - Think: Christopher Nolan / Denis Villeneuve framing

---

🎥 OUTPUT FORMAT (STRICT):

SCENE 1:
Description: [2 sentences, cinematic, environment-first, immersive]
Shot Type: [close-up | medium | wide]
Mood: [one word]
Image Prompt: [{char}, ...]

SCENE 2:
...

Continue until SCENE {num_scenes}.
"""
                gp = base_prompt
                if scene_focus:
                    gp = f"""SCENE-FIRST CINEMATIC PRINCIPLE:
Every image prompt must feel like a film still.

Order of thinking (STRICT):
1. Environment (specific real-world location)
2. Time of day
3. Weather / atmosphere
4. Lighting style (cinematic, dramatic, naturalistic)
5. Then introduce character action and emotion

The environment should NOT feel generic — it must be tangible, grounded, and visually rich.\n\n{base_prompt}"""

                resp       = client.models.generate_content(model="gemini-2.5-flash", contents=gp)
                story_text = resp.text
                st.success("Script generated")
                with st.expander("View Raw Script", expanded=False):
                    st.markdown(story_text)
            except Exception as e:
                st.error(f"Gemini error: {e}")
                st.stop()

        pat = (r'SCENE\s*(\d+):\s*\n'
               r'Description:\s*(.+?)\n'
               r'Shot Type:\s*(.+?)\n'
               r'Mood:\s*(.+?)\n'
               r'Image Prompt:\s*(.+?)(?=\nSCENE\s*\d+:|$)')
        scenes = re.findall(pat, story_text, re.DOTALL | re.IGNORECASE)

        if not scenes:
            pat2   = (r'SCENE\s*(\d+):\s*\n'
                      r'Description:\s*(.+?)\n'
                      r'Shot Type:\s*(.+?)\n'
                      r'Image Prompt:\s*(.+?)(?=\nSCENE\s*\d+:|$)')
            scenes = [(n, d, sh, "cinematic", p)
                      for n, d, sh, p in re.findall(pat2, story_text, re.DOTALL | re.IGNORECASE)]

        if not scenes:
            st.error("Could not parse scenes — review the raw script above.")
            st.stop()

        st.success(f"Parsed {len(scenes)} scenes")

        # ── Load pipeline
        if st.session_state.pipe is None:
            with st.spinner("Loading Stable Diffusion pipeline..."):
                dev = "cuda" if torch.cuda.is_available() else "cpu"
                vae = AutoencoderKL.from_pretrained(
                    "stabilityai/sd-vae-ft-mse", torch_dtype=torch.float16)
                pipe = StableDiffusionPipeline.from_pretrained(
                    "SG161222/Realistic_Vision_V5.1_noVAE",
                    vae=vae, torch_dtype=torch.float16, safety_checker=None).to(dev)
                pipe.scheduler = DPMSolverMultistepScheduler.from_config(
                    pipe.scheduler.config,
                    algorithm_type="dpmsolver++",
                    use_karras_sigmas=True)
                if dev == "cuda":
                    try: pipe.enable_xformers_memory_efficient_attention()
                    except: pass
                    pipe.enable_attention_slicing()
                    pipe.enable_vae_slicing()
                st.session_state.pipe            = pipe
                st.session_state.lora_loaded_path = None
                st.success("Pipeline loaded")

        pipe = st.session_state.pipe

        # ── Load LoRA
        lora_ok = False
        if lora_path and os.path.exists(lora_path):
            if st.session_state.lora_loaded_path != lora_path:
                with st.spinner("Loading LoRA weights..."):
                    try:
                        try: pipe.unload_lora_weights()
                        except: pass
                        pipe.load_lora_weights(
                            os.path.dirname(lora_path),
                            weight_name=os.path.basename(lora_path))
                        st.session_state.lora_loaded_path = lora_path
                        st.success(f"LoRA loaded: {os.path.basename(lora_path)}")
                    except Exception as e:
                        st.error(f"LoRA load failed: {e}")
                        import traceback; st.error(traceback.format_exc())
                        st.session_state.lora_loaded_path = None
            lora_ok = (st.session_state.lora_loaded_path == lora_path)
        else:
            st.warning("No LoRA loaded — using base model only.")

        dev  = "cuda" if torch.cuda.is_available() else "cpu"
        prog = st.progress(0)

        # ── Render scenes
        st.markdown("---")
        st.markdown('<div class="section-heading">Storyboard</div>', unsafe_allow_html=True)
        st.markdown(f'<div class="section-sub">{story_title} &nbsp;&mdash;&nbsp; {story_genre} &nbsp;&mdash;&nbsp; {len(scenes)} Scenes</div>', unsafe_allow_html=True)

        # Collect rendered images then display 2 per row
        rendered = []

        for idx, scene_data in enumerate(scenes):
            snum, sdesc, shot_raw, mood, iprompt = scene_data

            sf    = shot_raw.strip().lower()
            frame = "wide" if "wide" in sf else "medium" if "medium" in sf else "close" if "close" in sf else detect_frame(iprompt.strip())
            mood_str  = mood.strip().title() if mood.strip() else "Cinematic"
            shot_lbl  = {"wide": "Wide Shot", "medium": "Medium Shot", "close": "Close-Up"}[frame]
            shot_cls  = {"wide": "shot-wide",  "medium": "shot-med",   "close": "shot-close"}[frame]

            st.markdown(f"""
<div class="scene-card">
  <div class="scene-watermark">{snum}</div>
  <div class="scene-eyebrow">
    Scene {snum}
    <span class="shot-tag {shot_cls}">{shot_lbl}</span>
    <span class="shot-tag shot-mood">{mood_str}</span>
  </div>
  <div class="scene-description">{sdesc.strip()}</div>
  <div class="scene-mood-line">Mood &nbsp;&mdash;&nbsp; {mood_str}</div>
</div>
""", unsafe_allow_html=True)

            # Build prompts
            fpos, fneg = framing_tags(frame)
            qs         = quality_suffix(frame)
            raw        = iprompt.strip()

            if lora_ok:
                raw = strip_conflicting_identity(raw)
                raw = re.sub(re.escape(char), "", raw, flags=re.IGNORECASE).strip().lstrip(",").strip()
                raw = f"{char}, {raw}"
                if frame == "wide":
                    raw += f", {char} prominently visible in foreground, recognisable face, sharp detail"

            final_prompt   = f"{raw}, {fpos}, {qs}"
            final_negative = f"{COMMON_NEG}, {fneg}"
            guidance       = 7.0 if frame == "close" else 6.5
            eff_scale      = lscale if frame == "close" else min(lscale + 0.08, 1.4)

            with st.expander(f"Debug — Scene {snum}", expanded=False):
                st.write("**Frame:**", frame, " | **Mood:**", mood_str,
                         " | **LoRA:**", lora_ok, " | **LoRA scale:**", eff_scale if lora_ok else "N/A",
                         " | **Resolution:**", f"{gen_w}x{gen_h}")
                st.code(final_prompt, language="text")
                st.code(final_negative, language="text")

            with st.spinner(f"Rendering Scene {snum}..."):
                try:
                    gen = torch.Generator(dev).manual_seed(42 + idx)
                    kw  = dict(
                        prompt=final_prompt,
                        negative_prompt=final_negative,
                        width=gen_w, height=gen_h,
                        num_inference_steps=steps_g,
                        guidance_scale=guidance,
                        clip_skip=2,
                        generator=gen
                    )
                    if lora_ok and lscale > 0:
                        kw["cross_attention_kwargs"] = {"scale": eff_scale}
                    with torch.autocast(dev):
                        image = pipe(**kw).images[0]

                    if do_gfpgan or do_i2i:
                        with st.spinner(f"Enhancing faces — Scene {snum}..."):
                            if do_gfpgan:
                                image = restore_faces_gfpgan(image)
                            if do_i2i and frame in ("wide", "medium"):
                                char_prompt_for_i2i = (
                                    f"{char}, extremely detailed face, sharp eyes, "
                                    "natural skin, cinematic portrait"
                                )
                                image = sharpen_face_i2i(
                                    image, pipe, char_prompt_for_i2i,
                                    final_negative, steps_g, dev, lora_ok, lscale
                                )

                    os.makedirs("/content/generated", exist_ok=True)
                    safe_title = re.sub(r'[^a-zA-Z0-9_]', '_', story_title)
                    ipath      = f"/content/generated/scene_{snum}_{safe_title}.png"
                    image.save(ipath, "PNG")
                    rendered.append({
                        "num": snum, "label": shot_lbl, "mood": mood_str,
                        "path": ipath, "image": image,
                        "w": gen_w, "h": gen_h
                    })
                    prog.progress((idx + 1) / len(scenes))

                except Exception as e:
                    st.error(f"Scene {snum} failed: {e}")
                    import traceback; st.error(traceback.format_exc())

        # ── 2-per-row grid display
        st.markdown("---")
        st.markdown('<div class="section-heading">Rendered Frames</div>', unsafe_allow_html=True)
        st.markdown('<div class="section-sub">Two frames per row — download individually below each pair</div>', unsafe_allow_html=True)

        for i in range(0, len(rendered), 2):
            pair    = rendered[i:i+2]
            cols    = st.columns(2, gap="small")
            for col, sc in zip(cols, pair):
                with col:
                    st.image(
                        sc["image"],
                        caption=f"Scene {sc['num']}  |  {sc['label']}  |  {sc['mood']}  |  {sc['w']}x{sc['h']}",
                        use_container_width=True
                    )
                    with open(sc["path"], "rb") as fh:
                        st.download_button(
                            f"Download Scene {sc['num']}",
                            data=fh,
                            file_name=f"scene_{sc['num']}.png",
                            mime="image/png",
                            key=f"dl_{sc['num']}"
                        )
            st.markdown("")

        st.success(f"Storyboard complete — {len(rendered)} of {len(scenes)} scenes rendered.")
'''

with open("/content/streamlit_app.py", "w") as f:
    f.write(app_code)

from pyngrok import ngrok, conf

def start_streamlit():
    os.system("pkill -f streamlit")
    time.sleep(2)
    return subprocess.Popen(
        [sys.executable, "-m", "streamlit", "run", "/content/streamlit_app.py",
         "--server.port", "8501", "--server.headless", "true"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

def get_ngrok_tunnel():
    ngrok_token = "XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"
    conf.get_default().auth_token = ngrok_token
    ngrok.kill()
    time.sleep(1)
    return ngrok.connect(8501, bind_tls=True)

streamlit_process = start_streamlit()
time.sleep(5)
public_url = get_ngrok_tunnel()

print("=" * 60)
print("FRAMEFORGE — AI STORYBOARD STUDIO")
print("=" * 60)
print(f"Public URL: {public_url}")
print("=" * 60)

try:
    while True:
        if streamlit_process.poll() is not None:
            print("Streamlit crashed — restarting...")
            streamlit_process = start_streamlit()
            time.sleep(5)
        tunnels = ngrok.get_tunnels()
        if not tunnels:
            print("Ngrok disconnected — reconnecting...")
            public_url = get_ngrok_tunnel()
            print(f"New URL: {public_url}")
        time.sleep(30)
except KeyboardInterrupt:
    print("Shutting down...")
    ngrok.kill()
    streamlit_process.terminate()

/content
Installing dependencies...
Dependencies installed!
Cloning sd-scripts...
Installing sd-scripts requirements...
sd-scripts ready!
FRAMEFORGE — AI STORYBOARD STUDIO
Public URL: NgrokTunnel: "https://artier-adriel-lovably.ngrok-free.dev" -> "http://localhost:8501"
Shutting down...
